# Outils d'analyse des données trafic

## Nettoyage et agrégation par flag

In [2]:
import polars as pl
from datetime import datetime

def process_siredo_traffic(df: pl.DataFrame) -> pl.DataFrame:
    """
    Traite un DataFrame de données individuelles SIREDO (spm_bs_individuelle) :
    1. Ajout de flags de qualité (présence vs vitesse valide).
    2. Agrégation par station et par pas de temps (6 minutes).
    3. Calcul propre des débits et des vitesses moyennes.
    """
    
    # Étape 1 : Application des flags de qualification
    df_flagged = df.with_columns([
        # Flag 1 : Présence physique valide (exclut les erreurs fatales de capteurs : 
        # vitesses aberrantes hors limites physiques absolues ou longueurs négatives/nulles)
        (
            (pl.col("vitesse") >= 0) & 
            (pl.col("vitesse") <= 200) & 
            (pl.col("longueur") > 0)
        ).alias("is_valid_presence"),
        
        # Flag 2 : Vitesse exploitable pour les analyses statistiques 
        # (exclut les excès extrêmes ou les artefacts de couplage de boucles)
        (
            (pl.col("vitesse") >= 10) & 
            (pl.col("vitesse") <= 160)
        ).alias("is_valid_speed")
    ])
    
    # Étape 2 : Troncature temporelle et agrégation
    df_aggregated = (
        df_flagged
        .with_columns(
            # Découpage temporel standard SIREDO (ex: pas de 6 minutes)
            intervalle = pl.col("timestamp").dt.truncate("6m")
        )
        .group_by(["id_station", "intervalle"])
        .agg([
            # Débit brut (tous les enregistrements reçus)
            pl.len().alias("debit_brut"),
            
            # Débit réel/valide : on compte tous les véhicules ayant physiquement franchi 
            # la station (on ne supprime pas le véhicule du débit sous prétexte qu'il roule vite)
            pl.col("is_valid_presence").sum().alias("debit_valide"),
            
            # Vitesse moyenne calculée uniquement sur les véhicules dont la vitesse est valide
            pl.when(pl.col("is_valid_speed"))
              .then(pl.col("vitesse"))
              .mean()
              .alias("vitesse_moyenne"),
              
            # Indicateur de qualité : taux de rejet des vitesses sur l'intervalle
            (
                1.0 - (pl.col("is_valid_speed").sum() / pl.col("is_valid_presence").sum().clip(lower_bound=1))
            ).alias("taux_rejet_vitesse")
        ])
        .sort(["id_station", "intervalle"])
    )
    
    return df_aggregated


# ==========================================
# Exemple d'utilisation avec des données simulées
# ==========================================
if __name__ == "__main__":
    # Simulation d'un petit jeu de données SIREDO individuel
    data = {
        "id_station": ["STA_01", "STA_01", "STA_01", "STA_01", "STA_01"],
       "timestamp": [
            datetime(2026, 10, 14, 8, 0, 2),
            datetime(2026, 10, 14, 8, 0, 15),
            datetime(2026, 10, 14, 8, 2, 10),
            datetime(2026, 10, 14, 8, 5, 45),
            datetime(2026, 10, 14, 8, 6, 12),
        ],
        "vitesse": [135.0, 145.0, -1.0, 185.0, 90.0],  # -1 (erreur capteur), 185 (aberrant/technique)
        "longueur": [4.5, 16.2, 4.0, 5.0, 4.2],         # Véhicule 2 est un Poids Lourd
        "classe": ["VL", "PL", "VL", "VL", "VL"]
    }
    
    df_brut = pl.DataFrame(data)
    
    print("--- Données Brutes ---")
    print(df_brut)
    
    df_resultat = process_siredo_traffic(df_brut)
    
    print("\n--- Données Agrégées et Nettoyées (Pas de 6 min) ---")
    print(df_resultat)

--- Données Brutes ---
shape: (5, 5)
┌────────────┬─────────────────────┬─────────┬──────────┬────────┐
│ id_station ┆ timestamp           ┆ vitesse ┆ longueur ┆ classe │
│ ---        ┆ ---                 ┆ ---     ┆ ---      ┆ ---    │
│ str        ┆ datetime[μs]        ┆ f64     ┆ f64      ┆ str    │
╞════════════╪═════════════════════╪═════════╪══════════╪════════╡
│ STA_01     ┆ 2026-10-14 08:00:02 ┆ 135.0   ┆ 4.5      ┆ VL     │
│ STA_01     ┆ 2026-10-14 08:00:15 ┆ 145.0   ┆ 16.2     ┆ PL     │
│ STA_01     ┆ 2026-10-14 08:02:10 ┆ -1.0    ┆ 4.0      ┆ VL     │
│ STA_01     ┆ 2026-10-14 08:05:45 ┆ 185.0   ┆ 5.0      ┆ VL     │
│ STA_01     ┆ 2026-10-14 08:06:12 ┆ 90.0    ┆ 4.2      ┆ VL     │
└────────────┴─────────────────────┴─────────┴──────────┴────────┘

--- Données Agrégées et Nettoyées (Pas de 6 min) ---
shape: (2, 6)
┌────────────┬──────────────┬────────────┬──────────────┬─────────────────┬────────────────────┐
│ id_station ┆ intervalle   ┆ debit_brut ┆ debit_valide ┆ vit

## Calage des données Siredo pour tracer le diagramme fondamental de la station

In [ ]:
import polars as pl
import numpy as np
from scipy.optimize import curve_fit

# =====================================================================
# 1. DÉFINITION DES 4 MODÈLES MATHÉMATIQUES
# =====================================================================

def model_exp_pur(k, a, b, alpha):
    """1. Modèle Exponentiel Généralisé Pur"""
    return a * np.exp(-b * (np.maximum(0.0, k) ** alpha))

def model_exp_mixte(X, a, b, alpha, Cvp):
    """2. Modèle Exponentiel Généralisé Mixte (avec proportion p de PL)"""
    k, p = X
    num = a * np.exp(-b * (np.maximum(0.0, k) ** alpha))
    den = 1.0 + (Cvp - 1.0) * p
    return num / den

def model_pow_pur(k, a, b, alpha):
    """3. Modèle Puissance Généralisé Pur"""
    return a + b * (np.maximum(0.0, k) ** alpha)

def model_pow_mixte(X, a, b, alpha, Cvp):
    """4. Modèle Puissance Généralisé Mixte (avec proportion p de PL)"""
    k, p = X
    num = a + b * (np.maximum(0.0, k) ** alpha)
    den = 1.0 + (Cvp - 1.0) * p
    return num / den


# =====================================================================
# 2. PIPELINE DE PRÉPARATION ET CALAGE UNIFIÉ
# =====================================================================

def caler_modele_siredo(df_agg: pl.DataFrame, type_modele: str = "exp_mixte") -> dict:
    """
    Prépare les données SIREDO et calibre le modèle choisi parmi :
    - 'exp_pur' : V = a * exp(-b * K^alpha)
    - 'exp_mixte' : V = (a * exp(-b * K^alpha)) / (1 + (Cvp - 1) * p)
    - 'pow_pur' : V = a + b * K^alpha
    - 'pow_mixte' : V = (a + b * K^alpha) / (1 + (Cvp - 1) * p)
    """
    # Validation du type de modèle
    modeles_valides = ["exp_pur", "exp_mixte", "pow_pur", "pow_mixte"]
    if type_modele not in modeles_valides:
        raise ValueError(f"Modèle inconnu. Choisissez parmi : {modeles_valides}")

    # 1. Extraction en DataFrame Pandas et calculs préalables
    df_pd = df_agg.drop_nulls(subset=["debit_vl", "debit_pl", "vitesse_moyenne"]).to_pandas()
    
    # Conversion en débits horaires (Q_6min * 10)
    df_pd["debit_vl_h"] = df_pd["debit_vl"] * 10.0
    df_pd["debit_pl_h"] = df_pd["debit_pl"] * 10.0
    df_pd["debit_total_h"] = df_pd["debit_vl_h"] + df_pd["debit_pl_h"]
    
    # Proportion de PL (p) et Densité totale K (veh/km)
    df_pd["p"] = df_pd["debit_pl_h"] / df_pd["debit_total_h"].clip(lower_bound=1.0)
    df_pd["k"] = df_pd["debit_total_h"] / df_pd["vitesse_moyenne"]
    
    # Filtrage des aberrations et conditions limites
    mask = (df_pd["vitesse_moyenne"] > 10.0) & (df_pd["debit_total_h"] > 50.0)
    df_clean = df_pd[mask]
    
    k_data = df_clean["k"].values
    p_data = df_clean["p"].values
    v_data = df_clean["vitesse_moyenne"].values
    
    # 2. Configuration des paramètres selon le modèle
    if type_modele == "exp_pur":
        func = model_exp_pur
        xdata = k_data
        p0 = [130.0, 0.01, 1.0]
        bounds = ([90.0, 1e-6, 0.1], [150.0, 1.0, 3.0])
    elif type_modele == "exp_mixte":
        func = model_exp_mixte
        xdata = (k_data, p_data)
        p0 = [130.0, 0.01, 1.0, 2.5]
        bounds = ([90.0, 1e-6, 0.1, 1.0], [150.0, 1.0, 3.0, 6.0])
    elif type_modele == "pow_pur":
        func = model_pow_pur
        xdata = k_data
        p0 = [130.0, -0.5, 1.0]
        bounds = ([90.0, -100.0, 0.1], [150.0, 0.0, 3.0])
    elif type_modele == "pow_mixte":
        func = model_pow_mixte
        xdata = (k_data, p_data)
        p0 = [130.0, -0.5, 1.0, 2.5]
        bounds = ([90.0, -100.0, 0.1, 1.0], [150.0, 0.0, 3.0, 6.0])

    # 3. Exécution de l'optimisation (curve_fit)
    try:
        popt, pcov = curve_fit(func, xdata, v_data, p0=p0, bounds=bounds)
        
        # Prédiction et calcul du R2
        v_pred = func(xdata, *popt)
        ss_res = np.sum((v_data - v_pred) ** 2)
        ss_tot = np.sum((v_data - np.mean(v_data)) ** 2)
        r2 = 1.0 - (ss_res / ss_tot)
        
    except Exception as e:
        print(f"Erreur lors de l'optimisation pour le modèle '{type_modele}' : {e}")
        return {}

    # Restitution structurée des résultats
    resultat = {
        "modele": type_modele,
        "r2": r2,
        "covariance": pcov
    }
    
    if "mixte" in type_modele:
        resultat["params"] = {"a": popt[0], "b": popt[1], "alpha": popt[2], "Cvp": popt[3]}
    else:
        resultat["params"] = {"a": popt[0], "b": popt[1], "alpha": popt[2]}
        
    return resultat